(expression)=
# 表达式

表达式（expression）是操作和转换数据帧的核心要素。Polars 的表达式定义“作用于列的计算”，本身不产生结果，交给 `select()`、`with_columns()` 这类方法时才真正求值。

正因为计算被推迟，Polars 才能把整条查询一起优化：合并扫描、剪掉无用列、复用重复计算。这一章从“怎么用表达式”讲到“表达式怎么选列、怎么命名、输出多长”，最后汇总常用表达式。

In [1]:
import polars as pl
import numpy as np
from polars import selectors as cs
from helper.jupyter import row, capture_except

## 使用表达式的方法

这里简要说明三个最常用的方法：`select` 决定输出哪些列，`with_columns` 决定在原表上增加什么，`filter` 决定保留哪些行。三者的区别只在“对结果做什么”，表达式本身的写法是一样的。

### select

`DataFrame.select()` 从数据帧中取出指定列，或返回表达式计算的结果：传给 `select()` 的每个表达式都会成为结果中的一列。

`select()` 和 `select_seq()` 都用原数据帧的数据来计算表达式，区别在于 `select()` 并行计算各表达式，而 `select_seq()` 按顺序计算。大多数情况下用 `select()` 就好；表达式之间有依赖、或者想省内存时，才用后者。

下面的例子演示两者的用法。

In [2]:
df = pl.DataFrame({
    "a": [1, 2, 3],
    "b": [4, 5, 6],
    "c": [7, 8, 9]
})

df1 = df.select(["a", pl.col("c") * 2])
df2 = df.select("a", pl.col("c") * 2)
df3 = df.select_seq(
    a = pl.col("a") * 2,         # "a" 乘以 2
    a_plus_c = pl.col("a") + pl.col("c") #❶
) 
row(df, df1, df2, df3)

shape: (3, 3) a b c i64 i64 i64 1 4 7 2 5 8 3 6 9 
 shape: (3, 2) a c i64 i64 1 14 2 16 3 18 
 shape: (3, 2) a c i64 i64 1 14 2 16 3 18 
 shape: (3, 2) a a_plus_c i64 i64 2 8 4 10 6 12

❶ 上例中，`select_seq()` 里计算 `a_plus_c` 列所用的 `pl.col("a")` 引用的是原数据的 `a` 列，而不是 `a` 列加倍后的结果。

### with_columns

`DataFrame.with_columns()` 的用法与 `DataFrame.select()` 类似，但会把新计算出的列和原有列一起返回，因此可以在已有数据帧上追加新列。

下面的例子演示用 `with_columns()` 添加列。`df1` 新增了 `a_plus_b` 和 `const` 两列；`df2` 更新了原有的 `a` 列，并新增了 `a_plus_c` 列。

In [3]:
df1 = df.with_columns(
    a_plus_b = pl.col("a") + pl.col("b"),  # 添加 "a" + "b" 的列
    const = 10  # 添加常数列
)

df2 = df.with_columns(
    a = pl.col("a") * 2,
    a_plus_c = pl.col("a") + pl.col("c")
)

row(df1, df2)

shape: (3, 5) a b c a_plus_b const i64 i64 i64 i64 i32 1 4 7 5 10 2 5 8 7 10 3 6 9 9 10 
 shape: (3, 4) a b c a_plus_c i64 i64 i64 i64 2 4 7 8 4 5 8 10 6 6 9 12

### filter

`DataFrame.filter()` 按表达式返回的布尔条件筛选行，只返回条件为 `True` 的行。下面是使用 `filter()` 的例子。

In [4]:
df1 = df.filter(pl.col("a") > 1)  # 选取 "a" > 1 的行
df2 = df.filter((pl.col("a") > 1) & (pl.col("b") < 6)) # 条件的 AND
df3 = df.filter((pl.col("a") >= 3) |  (pl.col("b") <= 4)) # 条件的 OR
df4 = df.filter(pl.col("a") > 1, pl.col("b") < 6) # 多个表达式等同于用 & 连接
row(df, df1, df2, df3, df4)

a,b,c
i64,i64,i64
1,4,7
2,5,8
3,6,9
a,b,c
i64,i64,i64
2,5,8
3,6,9
a,b,c
i64,i64,i64


## 列的选择

多数表达式都从选择列开始。选列的方法有几种，各自适合不同的场景。

* `pl.col()`：按列名或数据类型选择列，日常用得最多。
* `pl.nth()`：按位置选择第 N 列，只在列顺序固定时合适。
* `pl.selectors`：用更复杂的逻辑选择列，比如“所有整数列”“名字匹配某正则的列”。

下面分别说明。

### pl.col()

`pl.col()` 用于按列名或列的数据类型选择列。

- **按列名选择：**
  指定列名即可选出该列，列名以字符串给出。

In [5]:
df = pl.DataFrame({
    "a": [1], "b": [4.0], "c": [7], "x1":[10.0], "y1":[20.0], "x2":[4], "y2":[10], "xyz":[10.0]})
selected_df = df.select(pl.col("a"))
print(selected_df.columns)

['a']


- **按多个列名选择：**
  也可以在列表中指定多个列名。

In [6]:
selected_df = df.select(pl.col(["a", "b"]))
print(selected_df.columns)

['a', 'b']


- **按数据类型选择：**
  指定数据类型，即可选出该类型的所有列。
  

In [7]:
selected_df = df.select(pl.col(pl.Int64))
print(selected_df.columns)

['a', 'c', 'x2', 'y2']


### pl.nth()

`pl.nth()` 用于选择数据帧中第 N 列，索引从 0 开始。

- **选择第 N 列：**
  指定索引，选出该索引对应的列。

  

In [8]:
selected_df = df.select(pl.nth(0))
print(selected_df.columns)

['a']


- **按多个索引选择：**
  也可以在列表中指定多个索引，选出这些索引对应的列。  

In [9]:
selected_df = df.select(pl.nth([0, 2]))
print(selected_df.columns)

['a', 'c']


### pl.all()

`pl.all()` 用于选择数据帧中的所有列。需要整体操作数据帧，或与其他选择条件组合时很方便。

In [10]:
selected_df = df.select(pl.all())
print(selected_df.columns)

['a', 'b', 'c', 'x1', 'y1', 'x2', 'y2', 'xyz']


### pl.exclude()

`pl.exclude()` 用于排除指定的列，可以方便地得到去掉某些列的数据帧。

- **排除指定列：**

In [11]:
selected_df = df.select(pl.exclude("b"))
print(selected_df.columns)

['a', 'c', 'x1', 'y1', 'x2', 'y2', 'xyz']


- **排除多个列：**

In [12]:
selected_df = df.select(pl.exclude(["a", "c"]))
print(selected_df.columns)

['b', 'x1', 'y1', 'x2', 'y2', 'xyz']


### selectors

`polars.selectors`（导入为 `cs`）模块提供更高级的列选择：不是按名字一个个点，而是按“类型、模式、前缀”这类条件批量选。

列多、或者列名会变时，用选择器比手写列名列表可靠得多。下面介绍主要函数及其用法。

#### 按数据类型选择列

`cs.by_dtype()` 按指定的数据类型选择列。

In [13]:
selected_df = df.select(cs.by_dtype(pl.Int64))
print(selected_df.columns)

['a', 'c', 'x2', 'y2']


`cs.integer()` 选出数据帧中所有整数型的列。Polars 的整数型包括 `Int8`、`Int16`、`Int32`、`Int64`、`UInt8`、`UInt16`、`UInt32`、`UInt64` 等。

`cs.float()` 选出数据帧中所有浮点型的列。Polars 的浮点型有 `Float32` 和 `Float64`。

In [14]:
selected_df = df.select(cs.integer())
print(selected_df.columns)

['a', 'c', 'x2', 'y2']


In [15]:
selected_df = df.select(cs.float())
print(selected_df.columns)

['b', 'x1', 'y1', 'xyz']


#### `cs.matches()`

用正则表达式选择列名。Polars 2.0 的 `cs.by_name()` 只做完全匹配，按模式选择时用 `cs.matches()`。

In [16]:
# 以 x 开头的列，或以数字结尾的列
selected_df = df.select(cs.matches(r"^x|y\d$"))
print(selected_df.columns)

['x1', 'y1', 'x2', 'y2', 'xyz']


#### `cs.starts_with()` 与 `cs.ends_with()`

选择具有指定前缀和后缀的列。

In [17]:
selected_df = df.select(cs.starts_with("x"))
print(selected_df.columns)

['x1', 'x2', 'xyz']


In [18]:
selected_df = df.select(cs.ends_with("1"))
print(selected_df.columns)

['x1', 'y1']


#### 集合运算

把上面介绍的函数组合起来，可以做集合运算——选择器之间支持 `|`、`-`、`&`、`~` 四种运算符。

* 并集（`|`）：合并多个列选择，扩大选择范围。例如选取某些列与所有列的组合。
* 差集（`-`）：用于排除指定列。例如从所有列中去掉某些列。
* 交集（`&`）：选取同时满足多个条件的列。例如选取既符合某个模式、又属于某个数据类型的列。
* 补集（`~`）：选取不满足某个条件的列。例如选取不符合某个模式的列。

这四种运算可以嵌套组合，所以“除了这几列之外的所有数值列”这类需求也能一行写完。

下面的选择语句选取以 'x' 开头的列，再从中排除以 '2' 结尾的列。

In [19]:
selected_df = df.select(cs.starts_with('x') - cs.ends_with('2'))
print(selected_df.columns)

['x1', 'xyz']


下面的选择语句选取整数列或以 '1' 结尾的列。

In [20]:
selected_df = df.select(cs.integer() | cs.ends_with('1'))
print(selected_df.columns)

['a', 'c', 'x1', 'y1', 'x2', 'y2']


## 列与列之间的表达式

下面三个表达式是三种不同形式的列运算，区别在于“一次作用在几列上”。

* `e1` — 列与列的运算。`A` 与 `B` 逐元素相加，结果只有一列。
* `e2` — 两列与一列的运算。实际会计算 `A + B` 和 `B + B` 两个表达式，所以结果有两列。
* `e3` — 选择器与表达式的组合。把所有数值列与 `B` 相加，因此表达式数量取决于数据的列结构，写代码时并不知道会生成几列。

In [21]:
e1 = pl.col("A") + pl.col("B")
e2 = pl.col("A", "B") + pl.col("B")
e3 = cs.numeric() + pl.col("B")  # 选择器与表达式可以直接组合

print(e1)
print(e2)
print(e3)

[(col("A")) + (col("B"))]
[(cs.by_name('A', 'B', require_all=true)) + (col("B"))]
[(cs.numeric()) + (col("B"))]


下面的程序用 `LazyDataFrame.explain()` 显示表达式运算的内部结构。

In [22]:
df = pl.DataFrame(dict(A=[1, 2], B=[10, 20], C=[100, 200], X=[1, 1], Y=[2, 2]))
ldf = df.lazy()
query = ldf.with_columns(C=e1)
print(query.explain())

simple π 5/5 ["A", "B", "C", "X", "Y"]
   WITH_COLUMNS:
   [(col("A") + col("B")).alias("C")] 
    DF ["A", "B", "C", "X", ...]; PROJECT["A", "B", "X", "Y"] 4/5 COLUMNS


存在多列选择器时，会生成多个表达式。

In [23]:
query = ldf.select(e2)
print(query.explain())

SELECT [(col("A") + col("B")), (col("B") + col("B"))]
  DF ["A", "B", "C", "X", ...]; PROJECT["A", "B"] 2/5 COLUMNS


In [24]:
query = ldf.select(e3)
print(query.explain())

SELECT [(col("A") + col("B")), (col("B") + col("B")), (col("C") + col("B")), (col("X") + col("B")), (col("Y") + col("B"))]
  DF ["A", "B", "C", "X", ...]; PROJECT */5 COLUMNS


一个表达式中只能有一个多列选择器。例如下面的语句试图计算 `A - B` 和 `B - A`，但会报错。

In [25]:
%%capture_except
query = ldf.select(pl.col('A', 'B') - pl.col('B', 'A'))
print(query.explain())

SELECT [(col("A") - col("B")), (col("B") - col("A"))]
  DF ["A", "B", "C", "X", ...]; PROJECT["A", "B"] 2/5 COLUMNS


关于这个问题，GitHub 上有如下提案。

* [Allow expanding and renaming multiple columns ](https://github.com/pola-rs/polars/issues/13225)

目前可以像下面这样用 `pl.struct()` 把多个列合并成一个 `struct` 列再处理，从而实现多列之间的计算。

In [26]:
df.with_columns(
    (pl.struct('A', 'B') - pl.struct('B', 'A'))
       .struct.rename_fields(['AB', 'BA'])
       .struct.field('*')
)

A,B,C,X,Y,AB,BA
i64,i64,i64,i64,i64,i64,i64
1,10,100,1,2,-9,9
2,20,200,1,2,-18,18


## 表达式的输出名

向 `DataFrame` 添加新列或取回列计算结果时，通常要给结果命名。这个名字称为“表达式的输出名”。

输出名不只是标签：`with_columns()` 靠它决定“覆盖已有列”还是“新增一列”，多个表达式同名还会直接报错。所以理解输出名的规则，比记住 `alias()` 更重要。

### 根名与输出名

不指定输出名时，表达式中最先出现的列名就是输出名。表达式所引用的列名清单称为“根名”，可以用 `.meta.root_names()` 和 `.meta.output_name()` 查看。

In [27]:
def show_names(expr):
    print(f'root_names={expr.meta.root_names()}, output_name={expr.meta.output_name()}')

In [28]:
expr1 = pl.col("A") * pl.col("B") + 1
show_names(expr1)

root_names=['A', 'B'], output_name=A


In [29]:
expr2 = 1 + pl.col("B") * pl.col("A")
show_names(expr2)

root_names=['B', 'A'], output_name=literal


In [30]:
expr3 = (pl.col("A") * pl.col("B") + 1).alias("Z")

`output_name` 变成 `literal` 的情况，是表达式最左边不是列而是数值这类项的时候。用 `.name.keep()` 可以恢复成根名的第一个列名。

In [31]:
row(df.select(expr1), df.select(expr2), df.select(expr2.name.keep()), df.select(expr3))

A
i64
11
41
literal
i64
11
41
B
i64
11


多个表达式得到相同的输出名时，计算会报错。下面的 `10 + pl.col('A', 'B')` 会分别对 `A` 和 `B` 计算，但最左边是数值，两者输出名都是 `literal`，于是冲突。

In [32]:
%%capture_except
expr = 10 + pl.col('A', 'B')
df.select(expr)

DuplicateError: projections contained duplicate output name 'literal'. It's possible that multiple expressions are returning the same default column name. If this is the case, try renaming the columns with `.alias("new_name")` to avoid duplicate column names.


把列的顺序调换、让列位于最左边，或者像下面这样用 `.name.keep()` 把输出名改回 `root_names[0]`。

In [33]:
df.select(expr.name.keep())

A,B
i64,i64
11,20
12,30


用 `.name.suffix()` 或 `.name.prefix()` 可以在保留根名的前提下加前后缀。数值位于最左边的表达式名字会统一成 `literal`，所以这类方法要用在列在左侧的表达式中。

In [34]:
# 把列放在最左边，名字为 A_calc / B_calc
df.select((pl.col("A", "B") + 10).name.suffix("_calc"))

A_calc,B_calc
i64,i64
11,20
12,30


#### 调整根名的顺序

下面这个 `pl.when().then().otherwise()` 表达式中，`root_names` 里列名的顺序是 `then()`、`otherwise()`、`when()`。例如下面的表达式，`expr.meta.root_names` 为 `['x', 'y', 'j']`。

In [35]:
expr = pl.when(pl.col('j') > 50).then('x').otherwise('y')
print(expr.meta.root_names())

['x', 'y', 'j']


`when()` 中有多个列时，所有表达式的输出名都是 'x'，实际计算会因名字重复报 `DuplicateError`。

In [36]:
%%capture_except
cols = pl.col(['j', 'k', 'l'])
expr = pl.when(cols > 50).then('x').otherwise('y')
df.select(expr)

ColumnNotFoundError: "j" not found


这种情况下，可以在表达式开头加一个不影响计算结果、只影响根名顺序的表达式来解决。

In [37]:
df2 = pl.DataFrame(dict(
    j=np.random.randint(10, 99, 2),
    k=np.random.randint(10, 99, 2),
    l=np.random.randint(10, 99, 2),
    x=10,
    y=20,
))

df2.select(
    cols * 0 + pl.when(cols > 50).then('x').otherwise('y')
)

j,k,l
i32,i32,i32
20,20,10
20,20,20


### 指定输出名

#### `alias()`

用 `alias()` 方法给计算结果命名。

In [38]:
result = df.select((pl.col('A') + pl.col('B')).alias("A_plus_B"))
print(result.columns)

['A_plus_B']


#### 关键字参数

用 `select()`、`with_columns()` 等方法添加新列时，可以用关键字参数给该列命名。

In [39]:
result = df.select(A_plus_B=pl.col('A') + pl.col('B'))
print(result.columns)

['A_plus_B']


### 变换输出名

`.name` 下提供了若干操作输出名的方法。下面介绍常用的三个。

**`map`**

对根名套用函数来改变表达式的输出名。

In [40]:
result_df = df.select(
    (pl.col('A', 'B', 'C') + pl.col('B')).name.map(lambda name: name * 3)
)
print(result_df.columns)

['AAA', 'BBB', 'CCC']


**`prefix` 与 `suffix`**

在根列名的开头和结尾添加前缀或后缀。

In [41]:
result_df = df.select(
    (pl.col('A', 'B', 'C') + pl.col('B')).name.suffix("_res")
)
print(result_df.columns)

['A_res', 'B_res', 'C_res']


## 表达式的输出值

表达式对“一列”运算，但输出可以是一行、与原列等长、或者是任意长度——理解这一点，才能解释 `over()`、`agg()` 里那些看起来奇怪的结果。

### 表达式独立执行

用下面的程序说明 Polars 表达式独立执行的含义。

In [42]:
df = pl.DataFrame(
    dict(x=range(10), y=range(10))
)

result = df.select(pl.all().sample(4))
result

x,y
i64,i64
2,5
7,9
1,7
6,1


这个程序用 `pl.all()` 对数据帧 `df` 的所有列套用 `sample(4)`，从每列随机抽取 4 行。Polars 的表达式对 DataFrame 的各列独立作用，因此 `pl.all().sample(4)` 是各列各自随机选 4 行。虽然对每列做的操作相同，但执行是独立的，不同列选中的行不一定一致。

用 `seed` 参数可以让所有列的随机数使用同一个种子，即使独立执行也能取出同一行的数据。

In [43]:
df.select(pl.all().sample(4, seed=10))

x,y
i64,i64
1,1
6,6
0,0
7,7


另外，抽取随机行推荐直接用 `DataFrame.sample()`。

In [44]:
df.sample(4)

x,y
i64,i64
7,7
9,9
2,2
5,5


`.sort()` 对各个列的值独立排序。

In [45]:
df = pl.DataFrame(
    dict(x=[1, 3, 2], y=[9, 8, 7])
)

处理多列的表达式会先拆成处理单列的表达式，再独立执行。下面的例子分别对 x 列和 y 列排序。

In [46]:
df.select(pl.col('x', 'y').sort())

x,y
i64,i64
1,7
2,8
3,9


要成对排序，用 `DataFrame.sort()`。

In [47]:
row(df.sort('x', 'y'), df.sort('y', 'x'))

x,y
i64,i64
1,9
2,7
3,8
x,y
i64,i64
2,7
3,8
1,9


在表达式中做成对排序，用 `struct`。

In [48]:
df.select(
    pl.struct('y', 'x').sort().struct.field('*')
)

y,x
i64,i64
7,2
8,3
9,1


或者用 `sort_by()` 指定排序所依据的列。各列仍然独立运算，但决定顺序的列相同，所以结果中行与行的对应关系得以保留。

In [49]:
df.select(
    pl.all().sort_by("y", "x")
)

x,y
i64,i64
2,7
3,8
1,9


### 表达式的输出数据量

对长度为 N 的输入，表达式的输出长度有三种：1、N 和 X。

- 长度 1：聚合类表达式，整列压成一个值。
- 长度 N：逐元素运算，行数不变。
- 长度 X：筛选、抽样这类，行数由数据决定。

同一个 `select()` 里混用不同长度会报错，所以“聚合结果和原列放一起”时，得用 `over()` 或者先聚合再连接。

对长度为 N 的输入，表达式的输出长度有三种：1、N 和 X。下面分别举例说明。

#### 输出长度为 1

输出长度为 1 时，表达式对整列做聚合，返回单个结果，例如列的总和或平均值。下面的代码计算 `x` 列的总和，返回单个值 `45`。

In [50]:
df = pl.DataFrame({
    "x": range(5),
})

result = df.select(pl.col("x").sum())
result

x
i64
10


#### 输出长度为 N

输出长度为 N 时，表达式对每一行独立操作，返回与原数据帧等长的结果，例如对每个元素做标量运算。下面的代码给 `x` 列每个元素加 `1`，返回与原数据帧等长的结果。

In [51]:
result = df.select(pl.col("x") + 1)
result

x
i64
1
2
3
4
5


#### 输出长度为 X

输出长度为 X 时，某些操作会返回与原长度 N 不同的长度，例如抽样和筛选。下面的代码输出 x 列中大于等于 3 的值。

In [52]:
result = df.select(pl.col('x').filter(pl.col('x') >=3))
result

x
i64
3
4


#### 输出长度不同时的行为

长度为 1 的列，其值会被重复，扩展到与其他列相同的长度，从而保持 DataFrame 的一致性。

In [53]:
df = pl.DataFrame(
    dict(x=[1, 3, 2], y=[9, 8, 7])
)

df.select(
    x_mean = pl.col('x').mean(),
    x_plus_y = pl.col('x') + pl.col('y')
)

x_mean,x_plus_y
f64,i64
2.0,10
2.0,11
2.0,9


下面两个表达式的输出长度分别是 1 和 2。

In [54]:
df.select(
    x_mean = pl.col('x').mean(),
    x_top2 = pl.col('x').top_k(2)
)

x_mean,x_top2
f64,i64
2.0,2
2.0,3


一次操作中包含输出长度不同的多个列时，Polars 会报错。

In [55]:
%%capture_except
df.select(
    x_sq = pl.col('x')**2,
    x_top2 = pl.col('x').top_k(2)
)

ShapeError: Series length 2 doesn't match the DataFrame height of 3


同样的计算式，输出长度也会随列内容不同而变化。下面的表达式对 x 列输出长度为 2，对 y 列为 3，因此报错。

In [56]:
%%capture_except
df.select(
    pl.col('x', 'y').filter(pl.col('x', 'y') >= 2)
)

ShapeError: Series length 2 doesn't match the DataFrame height of 3


用 `Expr.implode()` 把多个值转成一个列表，所有表达式的输出长度都变成 1，可以避免报错。

In [57]:
df.select(
    pl.col('x', 'y').filter(pl.col('x', 'y') >= 2).implode()
)

x,y
list[i64],list[i64]
"[3, 2]","[9, 8, 7]"


## 表达式的上下文

同一个表达式，放在不同的方法里，拿到的数据不一样——这就是“上下文”。

`select()`、`with_columns()` 把整列交给表达式；`group_by().agg()` 把每个分组的列交给表达式；`.list.eval()` 把每个列表交给表达式。写表达式前先想清楚“我现在在哪个上下文里”，比事后调试省事。

表达式执行的环境（上下文）不同，交给它处理的数据也不同。`select()`、`with_columns()` 会把整列数据交给表达式；`group_by().agg()` 则把每个分组的列交给表达式。

In [58]:
df = pl.DataFrame(dict(x=[1, 2, 1, 2, 3, 3], y=[10, 20, 30, 40, 50, 60]))

### agg 上下文

下面 `.agg()` 中的表达式，处理的是按 x 列分开的三个分组的数据。表达式输出多个值时，以列表保存。

In [59]:
df.group_by('x').agg(
    y = pl.col('y'),
    y_mult_x = pl.col('y') * pl.col('x'),
    sum_y_mult_x = (pl.col('y') * pl.col('x')).sum()
)

x,y,y_mult_x,sum_y_mult_x
i64,list[i64],list[i64],i64
3,"[50, 60]","[150, 180]",330
2,"[20, 40]","[40, 80]",120
1,"[10, 30]","[10, 30]",40


(list_context)=
### list 上下文

`.list.eval()` 的表达式上下文是待处理的各个列表。下面的程序计算 `x` 列中每个列表的平均值，再用它调整各列表。

- **表达式❶**用 `list.mean()` 计算每个列表的平均值，再从各列表元素中减去该平均值。
- **表达式❷**用 `list.eval()` 处理 `x` 列中的每个列表，并套用表达式❸。
- **表达式❸**中 `pl.element()` 表示当前处理的列表元素，从该元素中减去整个列表的平均值。
- **表达式❹**中 `list.agg()` 对 `x` 列的每个列表执行聚合表达式，结果列不再是列表型，而是标量值。这个例子计算每个列表最大值与最小值之差。

In [60]:
df = pl.DataFrame(dict(x=[[1, 2], [3, 4, 5, 6], [7, 8, 9]]))
df.with_columns(
    x1 = pl.col('x') - pl.col('x').list.mean(), #❶
    x2 = pl.col('x').list.eval(                 #❷
        pl.element() - pl.element().mean()      #❸
    ),
    x3 = pl.col('x').list.agg(                  #❹
        pl.element().max() - pl.element().min()
    )
)

x,x1,x2,x3
list[i64],list[f64],list[f64],i64
"[1, 2]","[-0.5, 0.5]","[-0.5, 0.5]",1
"[3, 4, … 6]","[-1.5, -0.5, … 1.5]","[-1.5, -0.5, … 1.5]",3
"[7, 8, 9]","[-1.0, 0.0, 1.0]","[-1.0, 0.0, 1.0]",2


下面的代码从每个列表的元素中挑出大于该列表平均值的内容。这类处理用 `list.eval()` 实现很简洁；不用 `list.eval()` 时，很难对整体列表做逐元素的条件处理。

- **表达式❺**中 `pl.element()` 表示列表内的各元素。
- `pl.element().mean()` 计算当前处理的整个列表的平均值。
- `filter` 方法判断每个元素是否大于列表平均值，只保留满足条件的元素。

In [61]:
df.with_columns(
    x_large = pl.col('x').list.eval(
        pl.element().filter(pl.element() > pl.element().mean()) #❺
    )    
)

x,x_large
list[i64],list[i64]
"[1, 2]",[2]
"[3, 4, … 6]","[5, 6]"
"[7, 8, 9]",[9]


把表达式❺中的 `pl.element()` 换成具体的列表，能更直观地理解这个计算。比如对列表 `[7, 8, 9]` 做如下计算。

In [62]:
s = pl.Series([7, 8, 9])
s.filter(s > s.mean()).to_list()

[9]

- 这里 `s.mean()` 计算列表的平均值（`8.0`）。
- `s > s.mean()` 判断每个元素是否大于平均值，生成布尔数组 `[False, False, True]`。
- `filter` 方法只保留满足该布尔条件的元素（`9`）。

### field 上下文

`.struct.with_fields()` 中的表达式对 struct 列的各字段进行计算。

In [63]:
df = pl.DataFrame(
    {
        "coords": [{"x": 1, "y": 4}, {"x": 4, "y": 9}, {"x": 9, "y": 16}],
        "multiply": [10, 2, 3],
    }
)
df

coords,multiply
struct[2],i64
"{1,4}",10
"{4,9}",2
"{9,16}",3


下面 `.struct.with_fields()` 中的表达式用 `pl.field(name)` 表示 struct 列中 name 字段的数据。

In [64]:
df.select(
    pl.col('coords').struct.with_fields(
        dist = (pl.field('x') ** 2 + pl.field('y') ** 2).sqrt() * pl.col('multiply')      
    ).struct.field('*')
)

x,y,dist
i64,i64,f64
1,4,41.231056
4,9,19.697716
9,16,55.072679


这里汇总说明常用的表达式：运算符、条件分支、取值、拼接、排序、横向运算。它们大多是一两个方法的组合，看一遍有个印象，用的时候回来查即可。

### 运算符

`+`、`-`、`*`、`/` 等运算符都有对应的表达式方法。例如下面的代码用 `+` 运算符和 `add()` 方法求 `x` 列与 `y` 列之和，用 `>=` 运算符和 `ge()` 方法做列与数值的比较。

In [65]:
df = pl.DataFrame(dict(
    x=[1, 2, 3, 4],
    y=[40, 30, 20, 10]
))

df.with_columns(
    add1=pl.col('x') + pl.col('y'),
    add2=pl.col('x').add(pl.col('y')),
    cond1=pl.col('x') >= 3,
    cond2=pl.col('x').ge(3)
)

x,y,add1,add2,cond1,cond2
i64,i64,i64,i64,bool,bool
1,40,41,41,false,false
2,30,32,32,false,false
3,20,23,23,true,true
4,10,14,14,true,true


`&` 和 `|` 分别对应 `and_` 与 `or_` 方法。下面的代码用 `|` 运算符和 `or_()` 方法计算两个条件的逻辑或（OR）。

In [66]:
df.with_columns(
    cond1=(pl.col('x') > 3) | (pl.col('y') < 10),
    cond2=pl.col('x').gt(3).or_(pl.col('y').lt(10))
)

x,y,cond1,cond2
i64,i64,bool,bool
1,40,false,false
2,30,false,false
3,20,false,false
4,10,true,true


### 把值包装成表达式

`pl.lit()` 用于把值包装成表达式。

传入 `Series` 或 NumPy 数组时，会被解释为**含多个值的列**，其长度决定结果的行数。其他数据类型都按**标量**处理，并按结果行数自动重复同一个值。

因此下面的例子会生成 3 行的 DataFrame（`Series` 和 NumPy 数组的长度），而 `int`、`str`、`dict` 在每行重复同一个值。

另外，Python 的 `list` 和 `tuple` 不按“列方向的多值”处理，而是解释为“作为单个元素的列表（list 型）”，因此每行都放入同一个列表。`dict` 则按 `struct` 型处理，这点也需要注意。

结果就是 `series` 和 `array` 沿行方向展开，而 `list` 和 `tuple` 保留为“每行都有同一列表的列”，这一差异在表中体现出来。

In [67]:
pl.select(
    pl.lit(1).alias("int"),
    pl.lit("hello").alias("str"),
    pl.lit(dict(x=1, y=2)).alias("dict"),
    pl.lit([1, 2, 3]).alias("list"),
    pl.lit((1, 2, 3)).alias("tuple"),
    pl.lit(pl.Series([1, 2, 3])).alias("series"),
    pl.lit(np.array([1, 2, 3])).alias("array"),
)

int,str,dict,list,tuple,series,array
i32,str,struct[2],list[i64],list[i64],i64,i64
1,"""hello""","{1,2}","[1, 2, 3]","[1, 2, 3]",1,1
1,"""hello""","{1,2}","[1, 2, 3]","[1, 2, 3]",2,2
1,"""hello""","{1,2}","[1, 2, 3]","[1, 2, 3]",3,3


想把一个 `pl.lit()` 变成多列，通常有两种做法。

* **用字典列表的方法**
  把字典列表转成结构体（`struct`）型的 `Series`，再传给 `pl.lit()`，然后调用 `.struct.unnest()` 把结构体各字段展开成独立的列。

* **用列表的列表的方法**
  把列表的列表转成 `list` 型的 `Series`，用 `.list.to_struct(fields=...)` 转成结构体列，最后用 `.struct.unnest()` 一次生成多列。

这样经由 `struct` 型再调用 `.struct.unnest()`，即使只有一个 `pl.lit()` 也能高效生成多列。把外部数据或计算结果整体作为列追加时尤其方便。

In [68]:
pl.select(
    # 字典列表 → struct Series → 多列
    pl.lit(
        pl.Series(
            [{"x": 1, "y": 2}, {"x": 2, "y": 3}, {"x": 3, "y": 4}]
        )
    )
    .struct.unnest(),

    # 列表的列表 → list Series → struct Series → 多列
    pl.lit(
        pl.Series([[1, 2], [2, 3], [3, 4]])
    )
    .list.to_struct(fields=["a", "b"])
    .struct.unnest(),
)

x,y,a,b
i64,i64,i64,i64
1,2,1,2
2,3,2,3
3,4,3,4


### 数学函数

`sin`、`sqrt` 这类一元函数用对应的方法计算；`arctan2` 这类二元函数用全局函数计算。

In [69]:
df.with_columns(
    sin=pl.col('x').sin(),
    sqrt=pl.col('y').sqrt(),
    arctan=pl.arctan2(pl.col('y'), pl.col('x')),
)

x,y,sin,sqrt,arctan
i64,i64,f64,f64,f64
1,40,0.841471,6.324555,1.545802
2,30,0.909297,5.477226,1.504228
3,20,0.14112,4.472136,1.421906
4,10,-0.756802,3.162278,1.19029


### 条件分支

条件分支以 `pl.when(cond)` 为起点，之后组合 `.then()`、`.when()`、`.otherwise()` 等方法表达。例如下面的代码实现如下条件分支。

```python
if x > 4:
    result = 'very good'
elif x > 3:
    result = 'good'
elif x > 2:
    result = 'fair'
elif x > 1:
    result = 'bad'
else:
    result = 'very bad'
```

In [70]:
df.with_columns(
    pl.when(pl.col('x') > 4)
      .then(pl.lit('very good'))
      .when(pl.col('x') > 3)
      .then(pl.lit('good'))
      .when(pl.col('x') > 2)    
      .then(pl.lit('fair'))
      .when(pl.col('x') > 1)    
      .then(pl.lit('bad'))    
      .otherwise(pl.lit('very bad'))
      .alias('level')
)

x,y,level
i64,i64,str
1,40,"""very bad"""
2,30,"""bad"""
3,20,"""fair"""
4,10,"""good"""


`when-then-otherwise` 的各表达式并行执行，无论条件如何都会全部求值。因此传给 `then` 和 `otherwise` 的表达式必须对整列都成立。下面的代码把以 'D' 开头的字符串的后续部分转成整数，但对 `"HAB"`、`"HFF"` 这类不以 `"D"` 开头的值也会套用 `str.slice(1).str.to_integer()`，无法转成数值而报错。

[关于此事的 issue](https://github.com/pola-rs/polars/issues/17601)

In [71]:
%%capture_except
df_s = pl.DataFrame(dict(val=["D12", "D32", "HAB", "HFF", "D100"]))

df_s.select(
    pl.when(pl.col("val").str.starts_with("D"))
    .then(pl.col("val").str.slice(1).str.to_integer())
    .otherwise(None)
)

如下使用 `str.to_integer(strict=False)`，可以让无法转成整数的字符串变成 `null`，从而避免报错。

In [72]:
df_s1 = df_s.select(
    pl.when(pl.col("val").str.starts_with("D"))
    .then(pl.col("val").str.slice(1).str.to_integer(strict=False))
    .otherwise(None)
)

或者先用 `when-then-otherwise` 把不满足条件的值转成 `null`，再做数据转换。

In [73]:
df_s2 = df_s.select(
    pl.when(pl.col("val").str.starts_with("D"))
    .then("val")
    .otherwise(None)
    .str.slice(1)
    .str.to_integer()
)

row(df_s1, df_s2)

val
i64
12
32
null
null
100
val
i64
12
32


`.cut()` 是按边界值把数据分段并打标签的表达式。Polars 2.0 中已弃用，`bin_intervals()` 承担同样的作用。

- `breaks`（`bin_intervals` 的第 1 个参数）— N 个边界值
- `labels` — N+1 个标签
- 区间的闭合方式 — `cut()` 的 `left_closed=False` 是右闭区间，对应 `bin_intervals()` 默认（左闭）时的 `right_closed=True`

下面的代码给 `x` 列打标签。

- `right_closed=True`（相当于 `cut(left_closed=False)`）— 右侧闭区间
  - `x <= 1` → very bad
  - `1 < x <= 2` → bad
  - `2 < x <= 3` → fair
  - `3 < x <= 4` → good
  - `4 < x` → very good

- 默认（左闭）— 左侧闭区间
  - `x < 1` → very bad
  - `1 <= x < 2` → bad
  - `2 <= x < 3` → fair
  - `3 <= x < 4` → good
  - `4 <= x` → very good

In [74]:
labels = ["very bad", "bad", "fair", "good", "very good"]
df.with_columns(
    level_right=pl.col("x").bin_intervals([1, 2, 3, 4], labels=labels, right_closed=True),
    level_left=pl.col("x").bin_intervals([1, 2, 3, 4], labels=labels),
)

x,y,level_right,level_left
i64,i64,enum,enum
1,40,"""very bad""","""bad"""
2,30,"""bad""","""fair"""
3,20,"""fair""","""good"""
4,10,"""good""","""very good"""


`.clip()` 以指定的最小值和最大值为边界，把超出范围的值改为边界值。下面的例子对 `y` 列做不同的边界裁剪。

- `y1` 列：把 `y` 列的值限制在 20 到 30 之间。  
- `y2` 列：只设下界 30，比 30 小的值被裁剪为 30。  
- `y3` 列：只设上界 20，比 20 大的值被裁剪为 20。  
- `y4` 列：上下界都用表达式给出。此时每个元素各自计算边界值，并据此裁剪。

In [75]:
df.with_columns(
    y1=pl.col('y').clip(20, 30),
    y2=pl.col('y').clip(lower_bound=30),
    y3=pl.col('y').clip(upper_bound=20),
    y4=pl.col('y').clip(pl.col('x') * 10, pl.col('x') * 20)
)

x,y,y1,y2,y3,y4
i64,i64,i64,i64,i64,i64
1,40,30,40,20,20
2,30,30,30,20,30
3,20,20,30,20,30
4,10,20,30,10,40


### 元素的替换

`.replace()` 和 `.replace_strict()` 可以替换元素。替换前后的值可以传两个列表，也可以传一个字典。两者的区别如下：

- **`.replace()`**：只替换部分元素，结果保持原数据类型。
- **`.replace_strict()`**：替换所有元素，结果变为替换后的数据类型。

下面的代码中，`x` 列的值替换后变成字符串型（`x2` 列），`y` 列的值保持原数据类型（`y2` 列）。

In [76]:
df.with_columns(
    x2=pl.col('x').replace_strict({1:'very bad', 2:'bad', 3:'fair'}, default='unknown'),
    y2=pl.col('y').replace([10, 20], [100, 200]),
)

x,y,x2,y2
i64,i64,str,i64
1,40,"""very bad""",40
2,30,"""bad""",30
3,20,"""fair""",200
4,10,"""unknown""",100


### 数据的选取

用 Polars 的表达式方法可以按条件提取和处理数据，比如取头尾、按位置取、按条件筛。

用 Polars 的表达式方法可以按条件提取和处理数据。下面是常用方法及其用途。

| **方法** | **说明** |
|---|---|
| `.head(n)` | 取前 `n` 行。 |
| `.tail(n)` | 取后 `n` 行。 |
| `.slice(offset, n)` | 从指定偏移取 `n` 行。 |
| `.get(index)` | 取指定索引的值。 |
| `.gather(indices)` | 按指定的索引列表取行。 |
| `.gather_every(n, offset=0)` | 按指定步长间隔取行。 |
| `.filter(condition)` | 取满足条件的行。 |
| `.top_k(k)` | 取值最大的 `k` 行。 |
| `.top_k_by(col, k)` | 按指定列取值最大的 `k` 行。 |
| `.bottom_k(k)` | 取值最小的 `k` 行。 |
| `.bottom_k_by(col, k)` | 按指定列取值最小的 `k` 行。 |
| `.sample(n)` | 随机取 `n` 行。 |

In [77]:
df_res = df.select(
    head=pl.col("x").head(2),                       # 前 2 行
    tail=pl.col("x").tail(2),                       # 后 2 行
    slice=pl.col("x").slice(1, 2),                  # 从偏移 1 起取 2 行
    get=pl.col("x").get(3),                         # 索引 3 的值
    gather=pl.col("x").gather([0, 3]),              # 索引 0 和 3 的行
    gather_every=pl.col("x").gather_every(2, offset=1), # 偏移 1、每 2 步取一行
    filter=pl.col("x").filter(pl.col("y") // 10 % 2 == 0), # 按条件筛选
    top_k=pl.col("x").top_k(2),                     # 取前 2 个
    top_k_by=pl.col("x").top_k_by(pl.col("y"), 2),  # 按 y 列取前 2 个
    bottom_k=pl.col("x").bottom_k(2),               # 取后 2 个
    bottom_k_by=pl.col("x").bottom_k_by(pl.col("y"), 2), # 按 y 列取后 2 个
    sample=pl.col("x").sample(2),                   # 随机 2 行
)
row(df, df_res)

shape: (4, 2) x y i64 i64 1 40 2 30 3 20 4 10 
 shape: (2, 12) head tail slice get gather gather_every filter top_k top_k_by bottom_k bottom_k_by sample i64 i64 i64 i64 i64 i64 i64 i64 i64 i64 i64 i64 1 3 2 4 1 2 1 4 1 1 4 1 2 4 3 4 4 4 3 3 2 2 3 4

Polars 的许多方法不仅能接受普通值，也能接受表达式作为参数，从而实现灵活、动态的数据操作。例如给 `.get()` 传表达式，就能动态地取出多个元素。

下面的代码中，

- `pl.len()` 取得数据帧的行数。
- `(pl.col("x") + 1) % pl.len()` 给 `x` 列每个元素加 1，再对行数取余，作为索引。
- `.get()` 用上面算出的索引取元素。

In [78]:
df.with_columns(
    get=pl.col('y').get((pl.col('x') + 1) % pl.len())
)

x,y,get
i64,i64,i64
1,40,20
2,30,10
3,20,40
4,10,30


### 数据的拼接

用 `concat()` 或 `.append()` 可以把多个运算结果纵向拼接；用 `.extend_constant()` 可以把常数重复追加多次。

下面的程序演示把 `x` 列与 `y` 列的一部分或常数拼接。

In [79]:
df2 = df.select(
    append=pl.col('x').append(pl.col('y').slice(1, 2)),
    concat=pl.concat([pl.col('x'), pl.col('y').slice(1, 2)]),
    extend=pl.col('x').extend_constant(99, 2),
)
row(df, df2)

shape: (4, 2) x y i64 i64 1 40 2 30 3 20 4 10 
 shape: (6, 3) append concat extend i64 i64 i64 1 1 1 2 2 2 3 3 3 4 4 4 30 30 99 20 20 99

`concat_list()`、`concat_arr()` 和 `struct()` 可以把多个运算结果横向合并，分别生成 `List` 型、`Array` 型和 `Struct` 型的列。下面的程序把 `x` 列、`y` 列以及 `x` 与 `y` 的商横向合并。

- `List` 型和 `Array` 型要求所有元素的数据类型一致。
  - `concat_list()` 会自动向上转型。
  - 用 `concat_arr()` 时需要手动转成相同的数据类型。
- `Struct` 型中各字段可以有不同的数据类型。

In [80]:
df3 = df.select(
    list=pl.concat_list('x', pl.col('y') / pl.col('x')),
    arr=pl.concat_arr(pl.col('x').cast(pl.Float64), pl.col('y') / pl.col('x')),
    struct=pl.struct('x', pl.col('y') / pl.col('x')),
)

row(df, df3)

shape: (4, 2) x y i64 i64 1 40 2 30 3 20 4 10 
 shape: (4, 3) list arr struct list[f64] array[f64, 2] struct[2] [1.0, 40.0] [1.0, 40.0] {1,40.0} [2.0, 15.0] [2.0, 15.0] {2,15.0} [3.0, 6.666667] [3.0, 6.666667] {3,6.666667} [4.0, 2.5] [4.0, 2.5] {4,2.5}

用 `.explode()` 可以把 `List` 型和 `Array` 型列的元素按**行优先**纵向拼接；`.list.explode()` 和 `.arr.explode()` 做同样的处理。

下面的程序把上面结果中的 `list` 列和 `arr` 列的元素纵向拼接。要把 `Struct` 型列按行优先拼接时，先用 `concat_list()` 把 `.struct.unnest()` 的结果转成 `List` 型，再用 `.explode()` 纵向拼接；按列优先拼接时，用 `concat()` 合并 `.struct.unnest()` 的结果。

In [81]:
df4 = df3.select(
    pl.col('list').explode(), 
    pl.col('arr').explode(),
    struct_row=pl.concat_list(pl.col('struct').struct.unnest()).explode(),
    struct_col=pl.concat([pl.col('struct').struct.unnest()])
)
row(df3, df4)

shape: (4, 3) list arr struct list[f64] array[f64, 2] struct[2] [1.0, 40.0] [1.0, 40.0] {1,40.0} [2.0, 15.0] [2.0, 15.0] {2,15.0} [3.0, 6.666667] [3.0, 6.666667] {3,6.666667} [4.0, 2.5] [4.0, 2.5] {4,2.5} 
 shape: (8, 4) list arr struct_row struct_col f64 f64 f64 f64 1.0 1.0 1.0 1.0 40.0 40.0 40.0 2.0 2.0 2.0 2.0 3.0 15.0 15.0 15.0 4.0 3.0 3.0 3.0 40.0 6.666667 6.666667 6.666667 15.0 4.0 4.0 4.0 6.666667 2.5 2.5 2.5 2.5

### 顺序调整

顺序相关的操作（平移、反转、排序）在时序和窗口计算里用得很多：`shift()` 造“上一期”的值，`sort()` 决定聚合的先后，`reverse()` 常用来配合累计计算。

`.shift()` 把一列的值向上或向下移动 N 个位置。N 为正时向下移，为负时向上移。空出的位置用 `fill_value` 参数指定的值填充，默认是 `NULL`。要做循环移位时，先用 `.append()` 把开头或结尾的值接上，再移位。

In [82]:
df1 = df.with_columns(
    x_shift1=pl.col("x").shift(1),
    x_shift2=pl.col("x").shift(-2),
)

df2 = df.select(
    x_shift1=pl.col('x').tail(1).append(pl.col('x')).shift(1).tail(pl.len()),
    x_shift2=pl.col('x').append(pl.col('x').head(2)).shift(-2).head(pl.len())
)

row(df1, df2)

shape: (4, 4) x y x_shift1 x_shift2 i64 i64 i64 i64 1 40 null 3 2 30 1 4 3 20 2 null 4 10 3 null 
 shape: (4, 2) x_shift1 x_shift2 i64 i64 4 3 1 4 2 1 3 2

`.reverse()` 把列的值反向排列。

In [83]:
df.with_columns(x_reverse=pl.col('x').reverse())

x,y,x_reverse
i64,i64,i64
1,40,4
2,30,3
3,20,2
4,10,1


`.sort()` 和 `.sort_by()` 按大小关系排列列的值。`descending=True` 时降序，默认升序（`False`）。  

- `.sort()`：对指定的列按升序或降序排列。  
- `.sort_by()`：可以指定多个作为排序依据的表达式。

下面的代码把 `x` 列降序排列的结果加为 `x_sort` 列，把 `y` 列升序排列的结果加为 `y_sort` 列；再以 `y` 列与 24 的距离为基准，把 `x` 和 `y` 列升序排列，加为 `x_sort_by` 和 `y_sort_by` 列。

In [84]:
df.with_columns(
    pl.col('x').sort(descending=True).alias('x_sort'),
    pl.col('y').sort().alias('y_sort'),
    pl.all().sort_by((pl.col('y') - 24).abs()).name.suffix('_sort_by')
)

x,y,x_sort,y_sort,x_sort_by,y_sort_by
i64,i64,i64,i64,i64,i64
1,40,4,10,3,20
2,30,3,20,2,30
3,20,2,30,4,10
4,10,1,40,1,40


### NULL 处理

与 NULL 相关的表达式在 [NULL 处理](#null) 中详细说明。

### 横向运算

函数名中含 `horizontal` 的函数在多个列之间做横向运算。下面的代码列出这些函数。

In [85]:
for name in dir(pl):
    if 'horizontal' in name:
        print(name)

all_horizontal
any_horizontal
cum_sum_horizontal
max_horizontal
mean_horizontal
min_horizontal
sum_horizontal


下面的代码计算 `tag` 列以外所有列的总和，并把结果加为新列 `sum`。

In [86]:
df = pl.DataFrame(
    {
     'tag': ['a', 'b', 'a', 'b', 'c', 'e'],        
     'A': [5, 1, 4, 5, 4, 6],
     'B': [3, 1, 1, 6, 4, 9],
     'C': [5, 6, 3, 7, 0, 9],
     'D': [0, 8, 0, 1, 0, 5],
     'E': [7, 7, 6, 2, 2, 8],
    }
)

df.with_columns(
    pl.sum_horizontal(pl.exclude("tag")).alias("sum")
)

tag,A,B,C,D,E,sum
str,i64,i64,i64,i64,i64,i64
"""a""",5,3,5,0,7,20
"""b""",1,1,6,8,7,23
"""a""",4,1,3,0,6,14
"""b""",5,6,7,1,2,21
"""c""",4,4,0,0,2,10
"""e""",6,9,9,5,8,37


`horizontal` 函数用不上时，也可以用 `concat_list()` 把多个列转成一个列表列，再用 `list.xxx` 函数计算。下面的例子计算所有列（`tag` 列除外）的中位数，加为新列 `median`。

In [87]:
df.with_columns(
    pl.concat_list(pl.exclude("tag"))
    .list.median()
    .alias("median")
)

tag,A,B,C,D,E,median
str,i64,i64,i64,i64,i64,f64
"""a""",5,3,5,0,7,5.0
"""b""",1,1,6,8,7,6.0
"""a""",4,1,3,0,6,3.0
"""b""",5,6,7,1,2,5.0
"""c""",4,4,0,0,2,2.0
"""e""",6,9,9,5,8,8.0


`list.xxx` 表达不了的计算可以用 `list.eval()` 写。下面的代码取每行值中最大的两个，把它们的和放入 `sum_top_2`。

`list.eval()` 中的 `.sum()` 把整个列表的和作为 1 个元素的列表返回，所以用 `.list.first()` 取出。

In [88]:
df.with_columns(
    pl.concat_list(pl.exclude("tag"))
    .list.eval(pl.element().sort(descending=True).head(2).sum())
    .list.first()
    .alias("sum_top_2")
)

tag,A,B,C,D,E,sum_top_2
str,i64,i64,i64,i64,i64,i64
"""a""",5,3,5,0,7,12
"""b""",1,1,6,8,7,15
"""a""",4,1,3,0,6,10
"""b""",5,6,7,1,2,13
"""c""",4,4,0,0,2,8
"""e""",6,9,9,5,8,18
